# Exercise 4 — The known RV giants

*Covers notebooks 01–04. Difficulty: ★★★☆☆*

Radial-velocity surveys have found dozens of cold giants around bright stars within 20 pc. Which of them could the ESC image in reflected light within a day?

In [ ]:
import warnings
import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt
import esc_etc
from esc_etc import Simulation, use_lazuli_style

use_lazuli_style()
warnings.filterwarnings("ignore", module="synphot")

def frame_for(sim, max_frame=1000 * u.s):
    """Frame time to use: max_frame, or half the saturation time if that is shorter."""
    return min(u.Quantity(max_frame, u.s), 0.5 * sim.saturation_time())


def hours_to_snr(sim, snr=5, max_frame=1000 * u.s):
    """Integration time (hours) to reach snr with the current configuration; NaN above the speckle plateau."""
    return sim.get_int_time_for_snr(snr, frame_for(sim, max_frame))["int_time_continuous"][0].to_value(u.hr)

### 4.1 — The sample

Load `12_nearby_rv_giants.csv` (RV planets with $0.3 < M \sin i < 15\,M_J$, $a > 1$ AU, $d < 20$ pc,
host $V < 7$, from the NASA Exoplanet Archive). For each planet compute the maximum separation
$a/d$ in arcsec and the reflected-light contrast at quadrature, $C = A_g (R_p/a)^2 / \pi$ with
$A_g = 0.5$ and the archive radius (1 $R_J$ where missing).

In [ ]:
import pandas as pd
from pathlib import Path
csv = next(d / "data" / "12_nearby_rv_giants.csv" for d in Path.cwd().parents if (d / "data" / "12_nearby_rv_giants.csv").exists())
df = pd.read_csv(csv)
print(len(df), "planets")
df[["pl_name", "pl_orbsmax", "pl_bmassj", "pl_radj", "sy_dist", "sy_vmag", "st_spectype"]].head()

In [ ]:
# your code here

### 4.2 — A cell with a bug

The cell below computes the time to S/N 5 for every planet. It runs, but the numbers are absurd:
every planet is detected in seconds. Find the bug, fix it, and keep going.

In [ ]:
sim = Simulation.from_default(filter="R3", hoststarmag=5.0, hoststar_band="johnson_v", planetdeltamag=20, contrast=1e-8, ppgain=10, separation=0.4)
edge = sim.coronagraph._lamD.max() * (sim.coronagraph.primary_filter / sim.coronagraph.diameter_primary).to_value(u.arcsec, equivalencies=u.dimensionless_angles())

def time_to_snr5(V, contrast, sep, ppgain=10):
    if sep > edge:
        return np.nan                                             # beyond the tabulated dark hole
    sim.update(scene__star__mag=V, scene__planet__deltamag=2.5 * np.log10(contrast),
               coronagraph__separation=sep, coronagraph__ppgain=ppgain)
    h = hours_to_snr(sim)
    sim.reset()
    return h

df["t_hr"] = [time_to_snr5(r.sy_vmag, r.contrast, r.sep_arcsec) for r in df.itertuples()]
print(df[["pl_name", "sy_vmag", "sep_arcsec", "contrast", "t_hr"]].sort_values("t_hr").head(8).to_string(index=False))

### 4.3 — Inside the working angle

Add a column `in_wa` for planets between the 50 %-throughput IWA and the table edge. How many
planets reach S/N 5 within a day *and* sit in the working angle at ppgain 10? And at ppgain 30?

In [ ]:
# your code here

### 4.4 — The picture

Plot contrast against separation for all planets, coloured by the time to S/N 5 (log scale), with
the IWA, the table edge and the contrast floor marked, and the names on the points.

In [ ]:
# your code here

> 🤖 **Ask Jupyter AI:** *The separations above are maximum elongations for circular orbits. For the three planets with eccentricity above 0.3 in the table, compute the range of projected separations over the orbit (ignoring inclination) and say whether they ever leave the working angle.*
>
> Then ask it to draft the observing-calendar extension: separation and phase angle on real dates, from an astrometric orbit.

***
## Share-back and feedback

Bring back **one plot and one number** to the share-back board, and the observing decision you draw
from them.

> 🤖 **Ask Jupyter AI:** *Draft a GitHub issue for schmidt-observatories/esc-etc describing one thing I
> found missing, confusing or wrong in this exercise, with a minimal code example.*